# Ch.6 — Production Ensembles

## EnsembleAI Challenge — Ch.6

The final chapter: **deploy** ensembles in production.

Key questions:
- Is the accuracy gain worth the latency cost?
- How many trees do we actually need?
- When should we use an ensemble vs a single model?

## Setup & Data

In [ ]:
# ── Setup & Data ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

IMG = "img/"
import os; os.makedirs(IMG, exist_ok=True)

np.random.seed(42)

data = fetch_california_housing()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_tr):,}  Test: {len(X_te):,}")

## Latency Benchmark

In [ ]:
# ── Latency Benchmark ────────────────────────────────────────────────────────
model_configs = {
    'Ridge': Ridge(alpha=1.0),
    'XGBoost (500)': XGBRegressor(n_estimators=500, max_depth=4,
                                    learning_rate=0.05, random_state=42, verbosity=0),
    'RF (200)': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1),
}

benchmark_results = {}
for name, model in model_configs.items():
    model.fit(X_tr, y_tr)
    _ = model.predict(X_te[:1])  # warm up
    
    latencies = []
    for i in range(1000):
        t0 = time.perf_counter()
        _ = model.predict(X_te[i % len(X_te):i % len(X_te) + 1])
        latencies.append((time.perf_counter() - t0) * 1000)
    
    rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te)))
    p50 = np.percentile(latencies, 50)
    p99 = np.percentile(latencies, 99)
    benchmark_results[name] = {'rmse': rmse, 'p50': p50, 'p99': p99}
    print(f"{name:>20}: RMSE={rmse:.4f}  P50={p50:.3f}ms  P99={p99:.3f}ms")

## Accuracy vs Latency Trade-off

In [ ]:
# ── Accuracy vs Latency Plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

for name, res in benchmark_results.items():
    ax.scatter(res['p50'], res['rmse'], s=150, zorder=5)
    ax.annotate(name, (res['p50'], res['rmse']),
                textcoords='offset points', xytext=(10, 5), fontsize=9)

ax.set_xlabel('Inference Latency P50 (ms)'); ax.set_ylabel('Test RMSE')
ax.set_title('Accuracy vs Latency Trade-off', fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch06_accuracy_vs_latency.png', dpi=150, bbox_inches='tight')
plt.show()

## Tree Pruning: How Many Trees Do We Need?

In [ ]:
# ── XGBoost Tree Pruning ─────────────────────────────────────────────────────
xgb_full = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                          random_state=42, verbosity=0)
xgb_full.fit(X_tr, y_tr)

tree_counts = [10, 25, 50, 100, 150, 200, 300, 500]
pruned_rmses = []
pruned_latencies = []

for n in tree_counts:
    preds = xgb_full.predict(X_te, iteration_range=(0, n))
    pruned_rmses.append(np.sqrt(mean_squared_error(y_te, preds)))
    
    lats = []
    for i in range(200):
        t0 = time.perf_counter()
        _ = xgb_full.predict(X_te[i:i+1], iteration_range=(0, n))
        lats.append((time.perf_counter() - t0) * 1000)
    pruned_latencies.append(np.median(lats))

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(tree_counts, pruned_rmses, 'o-', color='steelblue', linewidth=2, label='RMSE')
ax1.set_xlabel('Number of Trees Used'); ax1.set_ylabel('Test RMSE', color='steelblue')

ax2 = ax1.twinx()
ax2.plot(tree_counts, pruned_latencies, 's--', color='coral', linewidth=2, label='P50 Latency')
ax2.set_ylabel('P50 Latency (ms)', color='coral')

ax1.set_title('Tree Pruning: RMSE vs Latency', fontweight='bold')
ax1.grid(alpha=0.3)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
fig.savefig(f'{IMG}ch06_tree_pruning.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'Trees':>6} | {'RMSE':>8} | {'P50 (ms)':>10}")
print("-" * 30)
for n, r, l in zip(tree_counts, pruned_rmses, pruned_latencies):
    print(f"{n:>6} | {r:>8.4f} | {l:>10.3f}")

## Model Serialization & Versioning

In [ ]:
# ── Model Versioning Demo ────────────────────────────────────────────────────
import json
import hashlib

# Save model
xgb_full.save_model('img/xgb_model_v1.json')

# Save metadata
data_hash = hashlib.sha256(X_tr.tobytes()).hexdigest()[:16]
metadata = {
    'model': 'XGBRegressor',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 4,
    'test_rmse': float(np.sqrt(mean_squared_error(y_te, xgb_full.predict(X_te)))),
    'test_r2': float(r2_score(y_te, xgb_full.predict(X_te))),
    'train_data_hash': data_hash,
    'n_train': len(X_tr),
    'n_test': len(X_te),
    'feature_names': list(data.feature_names),
}

with open('img/model_metadata_v1.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Model versioning checklist:")
print(f"  ✅ Model saved: img/xgb_model_v1.json")
print(f"  ✅ Metadata saved: img/model_metadata_v1.json")
print(f"  ✅ Data hash: {data_hash}")
print(f"  ✅ RMSE: {metadata['test_rmse']:.4f}")

## Decision Framework: Ensemble or Not?

In [ ]:
# ── Decision Framework ───────────────────────────────────────────────────────
scenarios = [
    {'name': 'Real-time pricing API',
     'latency_budget_ms': 5, 'data_size': '50k', 'tabular': True,
     'recommendation': 'Single LightGBM (fast, accurate)'},
    {'name': 'Batch credit scoring',
     'latency_budget_ms': 1000, 'data_size': '500k', 'tabular': True,
     'recommendation': 'Stacked XGBoost + RF + Ridge (maximize accuracy)'},
    {'name': 'Image classification',
     'latency_budget_ms': 50, 'data_size': '100k images', 'tabular': False,
     'recommendation': 'Neural network (not tabular — ensembles lose)'},
]

print(f"{'Scenario':<25} {'Budget':>8} {'Data':>10} {'Recommendation'}")
print("=" * 80)
for s in scenarios:
    print(f"{s['name']:<25} {s['latency_budget_ms']:>6}ms {s['data_size']:>10} {s['recommendation']}")

## Final EnsembleAI Progress Check

| # | Constraint | Status | Evidence |
|---|-----------|--------|----------|
| 1 | IMPROVEMENT >5% | ✅ | Ensembles beat all single models |
| 2 | DIVERSITY | ✅ | Bagging + boosting + stacking |
| 3 | EFFICIENCY <5× | ✅ | Latency benchmarked; pruning works |
| 4 | INTERPRETABILITY | ✅ | SHAP per-prediction explanations |
| 5 | ROBUSTNESS | ✅ | Versioning + monitoring + A/B testing |

**All 5 EnsembleAI constraints achieved!**

## Track Complete

| Ch. | Topic | Key Takeaway |
|-----|-------|-------------|
| 1 | Bagging & RF | Average decorrelated trees → ↓variance |
| 2 | Boosting | Sequential error correction → ↓bias |
| 3 | XGB/LGB/Cat | Industrial frameworks with regularization |
| 4 | SHAP | Per-prediction Shapley explanations |
| 5 | Stacking | Meta-learner over diverse base models |
| 6 | Production | Latency, pruning, versioning, A/B testing |

## Exercises

1. **Pruning experiment**: For the XGBoost model, find the minimum number of trees that achieves within 1% of the full 500-tree RMSE. How much latency does this save?
2. **A/B test simulation**: Split test data into 100 batches. For each batch, compute RMSE for single XGBoost vs stacked ensemble. In what % of batches does the stack win?
3. **Feature drift detection**: Add Gaussian noise to MedInc in test data (simulating drift). Plot RMSE degradation as noise increases. How sensitive is XGBoost vs Ridge?

In [ ]:
# ── Exercise 1: Minimum trees within 1% ─────────────────────────────────────
# TODO: full_rmse = pruned_rmses[-1]
#     threshold = full_rmse * 1.01
#     Find smallest n where RMSE < threshold
pass

In [ ]:
# ── Exercise 2: A/B test simulation ──────────────────────────────────────────
# TODO: split X_te into 100 batches
#     For each batch: compute RMSE for XGBoost vs stack
#     Count wins
pass

In [ ]:
# ── Exercise 3: Feature drift sensitivity ───────────────────────────────────
# TODO: for noise_level in [0, 0.5, 1, 2, 5]:
#     X_te_noisy = X_te.copy()
#     X_te_noisy[:, 0] += np.random.normal(0, noise_level, len(X_te))
#     compute RMSE for XGBoost and Ridge on noisy data
pass